## Tips

- **Start simple** — a single GRU with 64 units is a good first attempt.
  Only add complexity (stacking, dropout, bidirectional) if the simple model
  is clearly underfitting or overfitting.

- **Use many epochs** — RNNs on time series converge slowly. 10 or 20 epochs
  is almost never enough. Use **at least 100 epochs** and let early stopping
  decide when to stop. The best checkpoint is saved automatically by
  `ModelCheckpoint` — you will not miss the optimal point even if you
  train for too long.

- **Watch the train/val gap** — if train MAE is much lower than val MAE
  you are overfitting. Add dropout, reduce `hidden_size`, or increase
  early stopping patience to give regularization more time to work.

- **Use TensorBoard** — run `tensorboard --logdir runs` in your terminal
  to monitor train and val curves in real time. A healthy training curve
  shows both losses decreasing together. A diverging val curve means overfitting.

- **Learning rate matters** — if the loss is not decreasing after the first
  few epochs, try a lower learning rate (`1e-4` instead of `1e-3`).
  Use `ReduceLROnPlateau` to automatically decay the learning rate when
  val MAE stops improving.

- **The sklearn baseline is hard to beat** — `HistGradientBoosting` with
  explicit lag features is a very strong baseline for tabular time series.
  This is not a failure of the RNN — it reflects a fundamental difference
  between the two approaches: tree models get temporal information from
  hand-crafted features, while RNNs must learn it from raw sequences.
  Getting within 10 bikes/hour of the sklearn baseline (< 44 bikes/hour)
  is an excellent result.

In [ ]:
import os
import numpy as np
import joblib

from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn

from torch.utils.tensorboard import SummaryWriter

In [29]:
save_dir = "data/bike_processed"

# --- Load arrays ---
raw_data    = np.load(os.path.join(save_dir, "raw_data.npy"))
counts = np.load(os.path.join(save_dir, "counts.npy"))
count_stats = np.load(os.path.join(save_dir, "count_stats.npy"))
train_idx   = np.load(os.path.join(save_dir, "train_idx.npy"))
val_idx     = np.load(os.path.join(save_dir, "val_idx.npy"))
test_idx    = np.load(os.path.join(save_dir, "test_idx.npy"))
naive_mae   = np.load(os.path.join(save_dir, "naive_mae.npy"))
preprocessor = joblib.load(os.path.join(save_dir, "preprocessor_rnn.pkl"))

# --- Recover split sizes ---
num_train = len(train_idx)
num_val   = len(val_idx)
num_test  = len(test_idx)

# --- Count stats in the training set ---
count_mean  = count_stats[0]
count_std   = count_stats[1]

# --- Sanity check ---
print(f"raw_data shape:    {raw_data.shape}")
print(f"counts shape: {counts.shape}")
print(f"train/val/test:    {num_train} / {num_val} / {num_test}")
print(f"counts range: {counts.min():.0f} to {counts.max():.0f} bikes/hour")
print(f"\nNaive baseline — Val MAE: {naive_mae[0]:.2f} | Test MAE: {naive_mae[1]:.2f} bikes/hour")

raw_data shape:    (17210, 28)
counts shape: (17210,)
train/val/test:    12047 / 2581 / 2582
counts range: 1 to 977 bikes/hour

Naive baseline — Val MAE: 93.26 | Test MAE: 80.78 bikes/hour


In [23]:
count_norm = (counts - count_mean) / count_std

In [6]:
class TimeseriesDataset(Dataset):
    def __init__(self, data, targets, sequence_length, sampling_rate, 
                 start_index, end_index, shuffle=False):
        self.data            = data
        self.targets         = targets
        self.sequence_length = sequence_length
        self.sampling_rate   = sampling_rate

        # Valid starting indices: each sequence of length sequence_length
        # sampled every sampling_rate steps needs:
        # (sequence_length - 1) * sampling_rate + 1 rows ahead
        self.indices = np.arange(start_index, end_index)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start = self.indices[idx]
        # Sample every `sampling_rate` steps for `sequence_length` steps
        steps = np.arange(start, start + self.sequence_length * self.sampling_rate, 
                          self.sampling_rate)
        x = self.data[steps]
        # Target is `delay` steps ahead of the sequence start
        y = self.targets[start + delay]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

In [22]:
sampling_rate   = 1    # already hourly
sequence_length = 24   # look back 24 hours
delay           = 1    # predict 1 hour ahead
batch_size      = 256

train_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=0, end_index=num_train,
)
val_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train, end_index=num_train + num_val,
)
test_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train + num_val, end_index=len(raw_data) - delay,
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

# --- Sanity check ---
inputs, targets = next(iter(train_loader))
print(f"Input shape:  {inputs.shape}")   # (256, 24, num_features)
print(f"Target shape: {targets.shape}")  # (256,)
print(f"Target range: {targets.min():.0f} to {targets.max():.0f}")

Input shape:  torch.Size([256, 24, 28])
Target shape: torch.Size([256])
Target range: -1 to 4
